# 데이터 불확실성과 모델 불확실성 실습

**Aleatoric · Epistemic Uncertainty**

측정 잡음처럼 데이터를 늘려도 남는 불확실성과, 데이터가 부족해서 생겨 보강하면 줄어드는 불확실성의 구분.

소재 분야에서 이해하기: 측정 반복으로 줄지 않는 산포와 미탐색 영역의 불확실성을 나눠 본다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 가우시안 프로세스 문서](https://scikit-learn.org/stable/modules/gaussian_process.html)

## 1. 두 종류의 불확실성 구분

데이터를 늘려서 줄어드는 불확실성과, 늘려도 남는 불확실성을 나눠봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

def experiment(n, noise=0.4, seed=0):
    local = np.random.default_rng(seed)
    x = local.uniform(0, 1, n)
    return x[:, None], np.sin(2 * np.pi * x) + local.normal(0, noise, n)

grid = np.linspace(0, 1, 200)[:, None]
print('관측 잡음 표준편차는 0.4로 고정입니다.')

In [ ]:
rows = []
for n in (10, 30, 100, 400, 1600):
    X_obs, y_obs = experiment(n)
    kernel = ConstantKernel(1.0) * RBF(0.2) + WhiteKernel(0.1)
    model = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0).fit(X_obs, y_obs)
    learned_noise = np.sqrt(model.kernel_.k2.noise_level * np.var(y_obs))
    _, total_std = model.predict(grid, return_std=True)
    epistemic = np.sqrt(np.maximum(total_std ** 2 - learned_noise ** 2, 0)).mean()
    rows.append((n, learned_noise, epistemic))
    print('n=%4d  데이터 불확실성(추정 잡음) %.3f  모델 불확실성 %.3f' % rows[-1])

rows = np.array(rows)
plt.plot(rows[:, 0], rows[:, 1], 'o-', label='aleatoric (noise)')
plt.plot(rows[:, 0], rows[:, 2], 's-', label='epistemic (model)')
plt.xscale('log'); plt.xlabel('samples'); plt.ylabel('standard deviation'); plt.legend(); plt.show()

## 2. 해석

모델 불확실성은 데이터를 늘리면 줄어듭니다. 측정 잡음에서 오는 불확실성은 데이터를 늘려도
일정 수준에 머무릅니다. 실험을 더 할지, 측정 방법을 개선할지 판단할 때 이 구분이 필요합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#aleatoric-epistemic)을 여세요.